# ПР-3. Пайплайн предобработки данных
**Проект:** CryptoMind AI
**Выполнил:** &lt;Баргыбаев Нуртилек CS32&gt;

## Задача
Собрать Pipeline: инженерия признаков (свой трансформер) → импутация →
масштабирование → модель. Работаем с BTC, цель — доходность дня t,
признаки строго из дней ≤ t−1 (никакой утечки — см. ПР-2).

In [6]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

df = pd.read_csv("../data/processed/crypto_daily.csv")
btc = df[df["symbol"] == "BTCUSDT"].sort_values("open_time").reset_index(drop=True)

# Цель: доходность дня t
y = np.log(btc["close"] / btc["close"].shift(1))

# Сырые входы для трансформера (только прошлое будет использоваться внутри)
X_raw = btc[["high", "low", "close", "quote_volume"]].iloc[1:]   # строка 0 без цели
y = y.iloc[1:]

## Свой трансформер
`CryptoFeatures` строит признаки дня t только из данных дней ≤ t−1:
лаг доходностей, вчерашний диапазон, скользящая волатильность, отношение объёмов.
Наследуем BaseEstimator (даёт get_params/set_params) и TransformerMixin
(делает объект совместимым с Pipeline).

In [7]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # добавляем корень проекта в путь

from src.features import CryptoFeatures

In [8]:
pipe = Pipeline([
    ("features", CryptoFeatures()),        # наш класс
    ("imputer", SimpleImputer(strategy="median")),   # NaN первых ~7 дней
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

# Хронологическое разбиение: сначала 80% — train, потом 20% — test
n = len(X_raw)
X_tr, X_te = X_raw.iloc[: int(n * 0.8)], X_raw.iloc[int(n * 0.8):]
y_tr, y_te = y.iloc[: int(n * 0.8)], y.iloc[int(n * 0.8):]

pipe.fit(X_tr, y_tr)
print("R² на train:", round(pipe.score(X_tr, y_tr), 4))
print("R² на test: ", round(pipe.score(X_te, y_te), 4))

R² на train: 0.0082
R² на test:  -0.0087


## Выводы
Pipeline собран, пропуски первых дней заполнены импьютером (обученным только
на train — утечки нет). R² около нуля ожидаем: гипотеза H1 из ПР-2 показала,
что дневная доходность близка к случайному блужданию. Это честный результат,
а не брак модели — выясним в ПР-4, насколько близко к нему подойдёт собственный
градиентный спуск.

In [9]:
import time

k = 30   # увеличиваем объём в 30 раз
rng = np.random.default_rng(42)
big_X = pd.concat(
    [X_raw.reset_index(drop=True) + rng.normal(0, 0.01, X_raw.shape)
     for _ in range(k)], ignore_index=True)
big_y = pd.concat([y.reset_index(drop=True)] * k, ignore_index=True)

t0 = time.perf_counter()
big_pipe = Pipeline(pipe.steps)   # свежая необученная копия пайплайна
big_pipe.fit(big_X, big_y)
dt = time.perf_counter() - t0

print(f"Объём: {len(big_X):,} строк (в {k} раз больше исходного)")
print(f"fit всего пайплайна: {dt:.2f} с ({len(big_X)/dt:,.0f} строк/с)")

Объём: 100,320 строк (в 30 раз больше исходного)
fit всего пайплайна: 0.16 с (614,175 строк/с)


## Тест производительности
Объём увеличен в 30 раз повторением данных с шумом (~100k строк).
Узкое место пайплайна — векторные shift/rolling в transform; операции
numpy/pandas векторизованы, поэтому даже на ~100k строк пайплайн остаётся
интерактивным (< 1 с). Дальнейший рост объёма без потерь возможен до
миллионов строк; при необходимости rolling-часть можно переписать на
numba, но здесь это не требуется.